In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')
#/content/drive/MyDrive/data1.1.tar.gz

In [ ]:
# import gdown
# url = "$SWISSDIAL"
# gdown.download(url, quiet=False)
# !tar -xvzf data1.1.tar.gz -C ./SwissDial

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
except:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print(e)

In [ ]:
from huggingface_hub import sync_bucket

sync_bucket(
    "hf://buckets/RobChio/swissgerman",
    "./bucket/"
)

In [ ]:
labels = {
    0: "AG",
    1: "BE",
    2: "BS",
    3: "GR",
    4: "LU",
    5: "SG",
    6: "VS",
    7: "ZH",
}

id2label = labels
label2id = {v: k for k, v in labels.items()}
num_labels = len(labels)

In [ ]:
#!pip install peft
from transformers import WhisperForConditionalGeneration, WhisperForAudioClassification, WhisperProcessor
#from peft import PeftModel, LoraConfig
import torch

model_id = "Flix-AI/flix-swissgerman-full"

processor = WhisperProcessor.from_pretrained(model_id)

# model = WhisperForConditionalGeneration.from_pretrained(
#     model_id, torch_dtype=torch.bfloat16, device_map="auto"
# )
model = WhisperForAudioClassification.from_pretrained(
    model_id,
    num_labels=8,
    label2id=label2id,
    id2label=id2label,
    torch_dtype=torch.bfloat16,
    #device_map="auto",
)

In [ ]:
# Freeze encoder
for param in model.parameters():
    param.requires_grad = False

# Train only the new classification layers
for param in model.projector.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
def extract_features(batch):
    audios = [sample["array"] for sample in batch["audio"]]
    
    # Generates standard 30-second (3000 frames) mel-spectrograms
    inputs = processor(
        audios,
        sampling_rate=16000,
        return_tensors="np"
        #return_tensors="pt"
    )
    
    labels = [
        label2id.get(code.upper(), code) if isinstance(code, str) else code
        for code in batch["dialect_code"]
    ]
    
    return {
        "input_features": inputs.input_features,  # Shape: (batch_size, 80, 3000)
        "labels": labels
    }

class FastFeatureCollator:
    def __call__(self, features):
        input_features = torch.stack([f["input_features"] for f in features]).to(torch.bfloat16)
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return {"input_features": input_features, "labels": labels}

In [ ]:
import datasets
from datasets import load_dataset, load_from_disk, Audio

try:
    train_dataset = load_from_disk("./bucket/preprocessed_swiss_train")
    eval_dataset = load_from_disk("./bucket/preprocessed_swiss_eval")
except Exception as e:
    print("Load the preprocessed the data first!")
    
train_dataset.set_format("torch")
eval_dataset.set_format("torch")

In [ ]:
%pip install evaluate
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
        
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./swissgerman-dialect-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=5,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    torch_compile=True, #False,
    dataloader_drop_last=False,
    hub_model_id="RobChio/swissgerman-dialect-classifier",
    push_to_hub=True,
    hub_private_repo=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=FastFeatureCollator(),
    compute_metrics=compute_metrics,
)

In [ ]:
# Train model
train_result = trainer.train()

# Log and save training state and metrics
# trainer.log_metrics("train", train_result.metrics)
# trainer.save_metrics("train", train_result.metrics)
# trainer.save_state()

In [ ]:
# eval_metrics = trainer.evaluate()
# trainer.log_metrics("eval", eval_metrics)
# trainer.save_metrics("eval", eval_metrics)

In [ ]:
# output_dir = "./dialect_classifier_model"

# # Save model, config, and label mappings
# trainer.save_model(output_dir)

# Save feature extractor and processor settings
# processor.save_pretrained(output_dir)

In [ ]:
# from google.colab import runtime
# runtime.unassign()